In [15]:
import sys, shutil, os, subprocess

print(f"Python version : {sys.version}")
print(f"Python path    : {sys.executable}")
print(f"Virtual env    : {sys.prefix != sys.base_prefix}")
print(f"pip available  : {shutil.which('pip') is not None}")
print(f"Working dir    : {os.getcwd()}")
print(f".env exists    : {os.path.isfile('.env')}")
print(f"gcloud CLI     : {shutil.which('gcloud') or 'Not found'}")

try:
    result = subprocess.run(["gcloud", "auth", "list", "--format=value(account)"],
                            capture_output=True, text=True, timeout=10)
    accounts = result.stdout.strip()
    print(f"GCP account    : {accounts if accounts else 'No active account'}")
except Exception:
    print("GCP account    : gcloud not available")

try:
    result = subprocess.run(["gcloud", "config", "get-value", "project"],
                            capture_output=True, text=True, timeout=10)
    print(f"GCP project    : {result.stdout.strip() or 'Not set'}")
except Exception:
    print("GCP project    : gcloud not available")

Python version : 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Python path    : c:\Program Files\Python314\python.exe
Virtual env    : False
pip available  : True
Working dir    : c:\Users\sjaysawal\OneDrive - Deloitte (O365D)\Desktop\project\gcp_labs\google_adk_gcp_labs
.env exists    : True
gcloud CLI     : Not found
GCP account    : gcloud not available
GCP project    : gcloud not available


In [17]:
gcloud auth login
gcloud auth application-default login
gcloud config set project YOUR_PROJECT_ID

SyntaxError: invalid syntax (1573760710.py, line 1)

In [16]:
# Google Maps Agent — Vertex AI + ADK
Directions, place search, and place details via Google Maps APIs.  
Configure `.env` with: `GOOGLE_MAPS_API_KEY`, `GOOGLE_CLOUD_PROJECT`, `GOOGLE_CLOUD_LOCATION`, `MODEL_ID`, `GOOGLE_GENAI_USE_VERTEXAI`

SyntaxError: invalid syntax (484647184.py, line 2)

In [18]:
!pip install google-adk google-genai requests python-dotenv --quiet

In [ ]:
import os, requests, asyncio, re
from typing import Optional
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

load_dotenv()
MAPS_KEY = os.environ["GOOGLE_MAPS_API_KEY"]
MODEL_ID = os.environ["MODEL_ID"]
print(f"Project: {os.environ['GOOGLE_CLOUD_PROJECT']} | Region: {os.environ['GOOGLE_CLOUD_LOCATION']} | Model: {MODEL_ID}")

Project : qwiklabs-gcp-00-7431166138f5
Region  : us-central1
Model   : gemini-2.0-flash-exp
Maps Key: ***nQi0


In [19]:
MAPS_BASE = "https://maps.googleapis.com/maps/api"

def _maps_get(endpoint: str, params: dict, timeout: int = 15) -> dict:
    """Helper: call a Maps API endpoint and return JSON or error dict."""
    params["key"] = MAPS_KEY
    r = requests.get(f"{MAPS_BASE}/{endpoint}", params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if data["status"] != "OK":
        return {"error": data.get("error_message", data["status"])}
    return data


def get_directions(origin: str, destination: str, mode: str = "driving") -> dict:
    """Get directions between two locations (driving/walking/bicycling/transit)."""
    data = _maps_get("directions/json", {"origin": origin, "destination": destination, "mode": mode})
    if "error" in data:
        return data
    leg = data["routes"][0]["legs"][0]
    strip = lambda s: re.sub(r"<[^>]+>", "", s)
    return {
        "origin": leg["start_address"], "destination": leg["end_address"],
        "distance": leg["distance"]["text"], "duration": leg["duration"]["text"], "mode": mode,
        "steps": [{"step": i, "instruction": strip(s["html_instructions"]),
                    "distance": s["distance"]["text"], "duration": s["duration"]["text"]}
                   for i, s in enumerate(leg["steps"], 1)],
    }


def search_places(query: str, location: Optional[str] = None) -> dict:
    """Search for places, restaurants, hotels, or attractions."""
    q = f"{query} in {location}" if location else query
    data = _maps_get("place/textsearch/json", {"query": q})
    if "error" in data:
        return data
    return {"query": q, "results": [
        {"name": p["name"], "address": p.get("formatted_address", "N/A"),
         "rating": p.get("rating", "N/A"), "types": p.get("types", [])[:3]}
        for p in data["results"][:5]
    ]}


def get_place_details(place_name: str) -> dict:
    """Get detailed info about a place (hours, phone, website, reviews)."""
    search = _maps_get("place/textsearch/json", {"query": place_name})
    if "error" in search or not search.get("results"):
        return {"error": f"Not found: {place_name}"}
    fields = "name,formatted_address,formatted_phone_number,website,rating,opening_hours,reviews,price_level,url"
    details = _maps_get("place/details/json", {"place_id": search["results"][0]["place_id"], "fields": fields})
    if "error" in details:
        return details
    p = details["result"]
    return {
        "name": p.get("name"), "address": p.get("formatted_address"),
        "phone": p.get("formatted_phone_number"), "website": p.get("website"),
        "rating": p.get("rating"), "google_maps_url": p.get("url"),
        "hours": p.get("opening_hours", {}).get("weekday_text", []),
        "reviews": [{"author": r.get("author_name"), "rating": r.get("rating"),
                      "text": r.get("text", "")[:200]} for r in p.get("reviews", [])[:3]],
    }

In [ ]:
agent = LlmAgent(
    name="google_maps_agent", model=MODEL_ID,
    description="Google Maps assistant for directions, place search, and place details.",
    instruction="You are a Google Maps assistant. Use get_directions for navigation, "
                "search_places to find POIs, get_place_details for detailed place info. "
                "Present results clearly.",
    tools=[get_directions, search_places, get_place_details],
)

runner = Runner(agent=agent, app_name="maps_app", session_service=InMemorySessionService())


async def ask(msg: str):
    """Send a query to the agent and print the response."""
    print(f"\n{'='*60}\nUSER: {msg}\n{'='*60}")
    content = types.Content(role="user", parts=[types.Part(text=msg)])
    resp = ""
    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=content):
        if event.is_final_response():
            resp = "".join(p.text for p in event.content.parts if p.text)
    print(f"\nAGENT:\n{resp}")
    return resp

Agent 'google_maps_agent' created with 3 tools


In [20]:
await ask("How do I get from Times Square to Central Park by walking?")

NameError: name 'ask' is not defined

In [ ]:
await ask("Find the best Italian restaurants in San Francisco")

In [ ]:
await ask("Tell me about the Eiffel Tower — hours, reviews, website")

In [ ]:
while (q := input("Ask (or 'quit'): ").strip()) and q.lower() not in ("quit", "exit", "q"):
    await ask(q)
print("Goodbye!")